### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
sys.path.append('./utils')

### Random seed for reproducibility

In [2]:
import torch
import random
import numpy as np
#import multiprocessing as mp
#mp.set_start_method("spawn", force=True)
# Now set your seed
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [3]:
import concurrent
import io
import logging
import re
import re2
import cairosvg
import kagglehub
from lxml import etree
from vllm import LLM, SamplingParams
import gc
import svg_constraints 
from svg_processor import SVGSanitizer, SVGProcessor

class Model:
    
    def __init__(self):

        self.model_path="./lora/Llama_32_3B_Instruct_lora_fp16_r256_s2000_i500_msl2048"
        self.model = LLM(
            model=self.model_path,
            max_model_len=1024,
            gpu_memory_utilization=0.85,
            dtype="half",
            seed=123,
            disable_log_stats=True
        )
       
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model 
        gc.collect()     


    def _format_prompt(self, description: str) -> str:
        return  f"""Below is an instruction that describes a task, paired with an input that provides further context. 
                Write a response that appropriately completes the request.
                
                ### Instruction:
                Generate a SVG code for the given input:
                
                ### Input:
                {description}
                
                ### Response:
                """
    
    def get_response(self, descriptions):
        
        formatted_input = [self._format_prompt(desc) for desc in descriptions]
        sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=1024,n=1)
        outputs = self.model.generate(formatted_input, sampling_params)
        
        #suitable for batch inputs as well
        output_list=[]
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
            output_list.append(generated_text.strip())
        return output_list
    
    def predict(self, descriptions: list[str], max_new_tokens=1024) -> list[str]:
        output_decoded_list = self.get_response(descriptions)
        final_svg_code_list = []
    
        for description, output in zip(descriptions, output_decoded_list):
            base_svg = SVGProcessor.clean_and_extract_svgs(output, self.default_svg)
            clean_svg = self.sanitizer.enforce_constraints(base_svg)
            final_svg = SVGProcessor.svg_conversion_check(description, clean_svg, self.default_svg)
            final_svg_code_list.append(final_svg)
    
        return final_svg_code_list


INFO 04-26 18:45:38 [__init__.py:239] Automatically detected platform cuda.


In [4]:
model=Model()

WARNING 04-26 18:45:39 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 04-26 18:45:43 [config.py:585] This model supports multiple tasks: {'classify', 'generate', 'embed', 'reward', 'score'}. Defaulting to 'generate'.
INFO 04-26 18:45:43 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-26 18:45:44 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/Llama_32_3B_Instruct_lora_fp16_r256_s2000_i500_msl2048', speculative_config=None, tokenizer='./lora/Llama_32_3B_Instruct_lora_fp16_r256_s2000_i500_msl2048', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=Deco

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 04-26 18:45:48 [loader.py:447] Loading weights took 2.73 seconds
INFO 04-26 18:45:48 [gpu_model_runner.py:1186] Model loading took 6.0160 GB and 2.880667 seconds
INFO 04-26 18:45:54 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/373df498e1/rank_0_0 for vLLM's torch.compile
INFO 04-26 18:45:54 [backends.py:425] Dynamo bytecode transform time: 5.89 s
INFO 04-26 18:45:55 [backends.py:115] Directly load the compiled graph for shape None from the cache
INFO 04-26 18:45:59 [monitor.py:33] torch.compile takes 5.89 s in total
INFO 04-26 18:46:00 [kv_cache_utils.py:566] GPU KV cache size: 17,952 tokens
INFO 04-26 18:46:00 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 17.53x
INFO 04-26 18:46:14 [gpu_model_runner.py:1534] Graph capturing finished in 14 secs, took 0.44 GiB
INFO 04-26 18:46:14 [core.py:151] init engine (profile, create kv cache, warmup model) took 25.70 seconds


In [5]:
#model.predict(['who are you?'])

In [6]:
#print(tmp[0])

In [7]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/svg_score_test_vqa.csv',header=[0])
print(df.shape)
df.head(2)

(75, 7)


,description,gpt_svg_1,gpt_score_sl,response,vqa_pair,response_2,gpt_svg_2
0,"'Vibrant autumn forest',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.904117,Here is the visual question answering (VQA) pa...,"{'description': 'Vibrant autumn forest', 'ques...","Here's an improved SVG representation of a ""Vi...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
1,"'Morning dew on grass',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.987024,Here is a visual question answering (VQA) pair...,"{'description': 'Morning dew on grass', 'quest...","Here's an improved SVG representation of ""Morn...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


In [8]:
# from tqdm import tqdm
# tqdm.pandas()
# df['svg_3'] = df['description'].progress_apply(lambda x: model.predict(x))

In [9]:
from tqdm import tqdm
description_list = [s.strip(" ',") for s in df['description'].to_list()]
batch_size = 12
results = []

for i in tqdm(range(0, len(description_list), batch_size), desc="Batch prediction"):
    batch = description_list[i:i + batch_size]
    batch_result = model.predict(batch)  # Ensure this handles a list of inputs
    results.extend(batch_result)


Batch prediction:   0%|                                   | 0/7 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/12 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   8%| | 1/12 [00:02<00:29,  2.71s/it, est. speed input: 22.52
cessed prompts:  17%|▏| 2/12 [00:06<00:35,  3.55s/it, est. speed input: 17.82
cessed prompts:  25%|▎| 3/12 [00:06<00:17,  1.99s/it, est. speed input: 26.21
cessed prompts:  33%|▎| 4/12 [00:07<00:10,  1.28s/it, est. speed input: 33.87
cessed prompts:  42%|▍| 5/12 [00:07<00:06,  1.07it/s, est. speed input: 40.82
cessed prompts:  50%|▌| 6/12 [00:08<00:06,  1.12s/it, est. speed input: 40.92
cessed prompts:  58%|▌| 7/12 [00:09<00:04,  1.12it/s, est. speed input: 45.44
cessed prompts:  67%|▋| 8/12 [00:11<00:04,  1.20s/it, est. speed input: 43.29
Processed prompts: 100%|█| 12/12 [00:13<00:00,  1.09s/it, est. speed input: 56.3
ERROR:root:SVG Parse Error: Unescaped '<' not allowed in attributes values, line 1, column 2498 (<string>, line 1). Returning default SV

In [10]:
df['svg_3']=results

In [11]:
model.close_model()

In [12]:
from siglip_class import SVGMetricEvaluator
from aesthetic_evaluator import AestheticEvaluator

In [13]:
#SigLip Score
from tqdm import tqdm
tqdm.pandas()
evaluator = SVGMetricEvaluator()
df['svg_score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg_3']), axis=1)

100%|███████████████████████████████████████████| 75/75 [00:04<00:00, 15.61it/s]


In [14]:
#Aes Score
from tqdm import tqdm
tqdm.pandas()
aes_eval = AestheticEvaluator()
df['aes_score_3'] = df.progress_apply(lambda row: aes_eval.get_score(row['svg_3']), axis=1)

100%|███████████████████████████████████████████| 75/75 [00:08<00:00,  9.23it/s]


In [15]:
#combined score
df['combined_score_3'] = (df['svg_score_3']+df['svg_score_3']+df['aes_score_3'])/3

In [16]:
print('mean_svg_score:',df['svg_score_3'].mean(),'mean_aes_score:',df['aes_score_3'].mean(),'combined_score:',df['combined_score_3'].mean())

mean_svg_score: 0.5017062060247454 mean_aes_score: 0.4608983497619628 combined_score: 0.48810358727048464


In [17]:
default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
df_default_svg=df[df['svg_3']==default_svg]
print('default_svg_count:',df_default_svg.shape[0])
print('default_svg_score_mean:',df_default_svg['svg_score_3'].mean(),'default_aes_score_mean:',df_default_svg['aes_score_3'].mean(),\
     'combined_score:',df_default_svg['combined_score_3'].mean())

default_svg_count: 21
default_svg_score_mean: 1.7350670191865598e-07 default_aes_score_mean: 0.4369946956634522 combined_score: 0.14566501422561864


In [18]:
df_non_default_svg=df[df['svg_3']!=default_svg]
print('non-default_svg_count:',df_non_default_svg.shape[0])
print('non-default_svg_score_mean:',df_non_default_svg['svg_score_3'].mean(),\
      'non-default_aes_score_mean:',df_non_default_svg['aes_score_3'].mean(),\
        'combined_score:',df_non_default_svg['combined_score_3'].mean())

non-default_svg_count: 54
non-default_svg_score_mean: 0.69681410755954 non-default_aes_score_mean: 0.470194215244717 combined_score: 0.621274143454599
